# 🎨 ComfyUI Google Colab 混合分层版（v3 修复版：Bypass LoRA QKV 修复 + 多 LoRA 叠加 + 云盘秒同步）

本 Notebook 采用**混合分层架构**：
- **Colab 本地高速 SSD（`/content/ComfyUI`）**：存放 ComfyUI 程序本体、ComfyUI-Manager 与 Python 依赖，启动与加载节点极快。
- **Google Drive 云盘（`/content/drive/MyDrive/ComfyUI_Data`）**：持久化保存**大模型权重、LoRA、VAE、插件清单（`custom_nodes_list.txt`）、工作流设置（`user/`）、输入图（`input/`）和生图结果（`output/`）**。

### 🐣 小白跑 `moss-image-sft-int8` 三步傻瓜流程
运行 MOSS-Image 需要 **3 个模型文件**（DiT 主模型 + Qwen3.5-9B 文本编码器 + FLUX.2 VAE），**Cell 3 已为你全部预置好一键下载**，并附带官方文字生图工作流 `image_moss_image_t2i.json`：
1. **先用 CPU 运行时下模型（不花 GPU 点数）**：按顺序点击运行 **Cell 1 → Cell 3**，等 3 个文件与工作流下载进 Google Drive。
2. **切换 GPU 开始跑图**：顶部菜单 `代码执行程序 → 更改运行时类型` 选 **L4 GPU**（或 T4 / A100），再依次运行 **Cell 1 → Cell 2 → Cell 4**（**跳过 Cell 3**）。
3. **打开网页加载工作流**：点击 Cell 4 输出的 `https://xxxx.trycloudflare.com` 链接进入 ComfyUI，点击左侧工作流/菜单里的 **`image_moss_image_t2i.json`**（或在 `Templates` 中搜索 `MOSS-Image`），确认模型名称已选对即可生图。

### ⚠️ 省点数与防风控 3 条原则
1. **确认余额大于 0**：启动前请在 Colab 右上角 `Settings -> Subscription` 确认有可用计算单元（0 点数免费档禁止运行外部 WebUI 隧道）。
2. **下模型用 CPU，跑图再开 GPU**：往云盘下载模型时用 CPU 即可；只有真正进画布出图时才开 L4 / T4 / A100。
3. **用完务必销毁运行时**：出图结束后，点击右上角 **断开连接并删除运行时（Disconnect and delete runtime）**，避免闲置继续扣点。

In [ ]:
#@title 1. 挂载 Google Drive 并初始化 5TB 云盘持久化目录
import os
import shutil
import subprocess
from google.colab import drive

DRIVE_DATA_DIR = "/content/drive/MyDrive/ComfyUI_Data" #@param {type:"string"}

print("正在挂载 Google Drive...")
drive.mount("/content/drive")

subdirs = [
    "models/checkpoints",
    "models/diffusion_models",
    "models/unet",
    "models/text_encoders",
    "models/clip",
    "models/clip_vision",
    "models/vae",
    "models/loras",
    "models/controlnet",
    "models/upscale_models",
    "models/embeddings",
    "input",
    "output",
    "user/default/workflows",
]
for sub in subdirs:
    os.makedirs(os.path.join(DRIVE_DATA_DIR, sub), exist_ok=True)

nodes_list_path = os.path.join(DRIVE_DATA_DIR, "custom_nodes_list.txt")
if not os.path.exists(nodes_list_path):
    with open(nodes_list_path, "w", encoding="utf-8") as f:
        f.write(
            "# 每行填写一个 GitHub 插件仓库地址，Cell 2 启动时会自动克隆并安装依赖\n"
            "# 示例（去掉行首 # 即可生效）：\n"
            "# https://github.com/city96/ComfyUI-GGUF.git\n"
        )

print(f"\n✅ 持久化目录已就绪: {DRIVE_DATA_DIR}")
print(f"✅ 插件恢复清单路径: {nodes_list_path}")

gpu_name = ""
if shutil.which("nvidia-smi"):
    try:
        gpu_res = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True,
            text=True,
        )
        if gpu_res.returncode == 0:
            gpu_name = gpu_res.stdout.strip()
    except OSError:
        pass

if gpu_name:
    print(f"🖥️ 当前硬件 (GPU): {gpu_name}")
else:
    print("🖥️ 当前硬件: CPU（适合运行 Cell 3 高速下载模型，不消耗 GPU 点数）")

total, used, free = shutil.disk_usage("/content/drive/MyDrive")
gb = 1024 ** 3
print(
    f"☁️ Google Drive 空间: 总计 {total / gb:.1f} GB | "
    f"已用 {used / gb:.1f} GB | 可用 {free / gb:.1f} GB"
)

In [ ]:
#@title 2. 在 Colab 高速 SSD 安装 ComfyUI + Manager 并绑定云盘模型目录
import os
import shutil
import subprocess
import sys

DRIVE_DATA_DIR = "/content/drive/MyDrive/ComfyUI_Data" #@param {type:"string"}
UPDATE_COMFYUI = True #@param {type:"boolean"}
INSTALL_GGUF_NODE = True #@param {type:"boolean"}
INSTALL_SEEDVR2_NODE = False #@param {type:"boolean"}
COPY_MODELS_TO_LOCAL_SSD = True #@param {type:"boolean"}

COMFY_DIR = "/content/ComfyUI"
CUSTOM_NODES_DIR = os.path.join(COMFY_DIR, "custom_nodes")


def run_cmd(cmd, cwd=None):
    print(f"$ {' '.join(cmd)}")
    subprocess.run(cmd, cwd=cwd, check=True)


def ensure_repo(repo_url, dest_dir, update=False):
    if not os.path.exists(dest_dir):
        run_cmd(["git", "clone", "--depth", "1", repo_url, dest_dir])
    elif update:
        run_cmd(["git", "pull"], cwd=dest_dir)
    req_file = os.path.join(dest_dir, "requirements.txt")
    if os.path.exists(req_file):
        run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", req_file])


if not os.path.exists(DRIVE_DATA_DIR):
    raise RuntimeError(f"未检测到云盘目录 {DRIVE_DATA_DIR}，请先运行 Cell 1 挂载 Google Drive！")

if shutil.which("aria2c") is None:
    print("正在安装 aria2...")
    run_cmd(["apt-get", "update", "-qq"])
    run_cmd(["apt-get", "install", "-y", "-qq", "aria2"])

print("\n[1/4] 安装/更新 ComfyUI 本体...")
ensure_repo("https://github.com/Comfy-Org/ComfyUI.git", COMFY_DIR, update=UPDATE_COMFYUI)

print("\n[2/4] 安装/更新 ComfyUI-Manager...")
ensure_repo(
    "https://github.com/Comfy-Org/ComfyUI-Manager.git",
    os.path.join(CUSTOM_NODES_DIR, "ComfyUI-Manager"),
    update=UPDATE_COMFYUI,
)

installed_repos = {"ComfyUI-Manager"}
if INSTALL_GGUF_NODE:
    print("\n[3/4] 安装/更新 ComfyUI-GGUF 量化模型支持...")
    ensure_repo(
        "https://github.com/city96/ComfyUI-GGUF.git",
        os.path.join(CUSTOM_NODES_DIR, "ComfyUI-GGUF"),
        update=UPDATE_COMFYUI,
    )
    installed_repos.add("ComfyUI-GGUF")

if INSTALL_SEEDVR2_NODE:
    print("\n[可选插件] 安装/更新 ComfyUI-SeedVR2_VideoUpscaler...")
    ensure_repo(
        "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git",
        os.path.join(CUSTOM_NODES_DIR, "ComfyUI-SeedVR2_VideoUpscaler"),
        update=UPDATE_COMFYUI,
    )
    installed_repos.add("ComfyUI-SeedVR2_VideoUpscaler")

nodes_list_path = os.path.join(DRIVE_DATA_DIR, "custom_nodes_list.txt")
if os.path.exists(nodes_list_path):
    with open(nodes_list_path, "r", encoding="utf-8") as f:
        extra_urls = [
            line.strip()
            for line in f
            if line.strip() and not line.strip().startswith("#")
        ]
    if extra_urls:
        print(f"\n检测到 {len(extra_urls)} 个自定义恢复插件，正在安装...")
        for url in extra_urls:
            repo_name = url.rstrip("/").split("/")[-1]
            if repo_name.endswith(".git"):
                repo_name = repo_name[:-4]
            if repo_name in installed_repos:
                continue
            ensure_repo(url, os.path.join(CUSTOM_NODES_DIR, repo_name), update=UPDATE_COMFYUI)
            installed_repos.add(repo_name)

print("\n[4/4] 生成 extra_model_paths.yaml 映射云盘模型目录...")
drive_base = DRIVE_DATA_DIR.rstrip("/") + "/"
yaml_content = f"""drive_models:
  base_path: {drive_base}
  is_default: true

  checkpoints: models/checkpoints/
  diffusion_models: |
    models/diffusion_models/
    models/unet/
  text_encoders: |
    models/text_encoders/
    models/clip/
  clip_vision: models/clip_vision/
  vae: models/vae/
  loras: models/loras/
  controlnet: models/controlnet/
  upscale_models: models/upscale_models/
  embeddings: models/embeddings/
"""
yaml_path = os.path.join(COMFY_DIR, "extra_model_paths.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

bridge_dir = os.path.join(CUSTOM_NODES_DIR, "moss_studio_bridge")
os.makedirs(bridge_dir, exist_ok=True)
bridge_code = r"""import asyncio
import hmac
import logging
import os
import re
import shutil
import urllib.request
import uuid
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

from aiohttp import web
from server import PromptServer

NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}

ALLOWED_SUBFOLDERS = {"loras", "upscale_models", "diffusion_models", "text_encoders", "vae"}
ALLOWED_EXTS = (".safetensors", ".pth", ".pt", ".ckpt")
DOWNLOAD_TASKS = {}

_MOSS_LAYER_PROJ_RE = re.compile(r"layers\.\d+\.proj\.")
_MOSS_EXTRA_PREFIXES = ("pipe.dit.", "dit.", "transformer.", "model.diffusion_model.")


def _install_bypass_lora_qkv_fix():
    try:
        import torch
        import comfy.lora
        import comfy.lora_convert
        import comfy.model_management
        import comfy.sd
        import comfy.weight_adapter
        from comfy.patcher_extension import PatcherInjection
        from comfy.weight_adapter.base import WeightAdapterBase, WeightAdapterTrainBase
        from comfy.weight_adapter.bypass import get_module_type_info
    except Exception as exc:
        logging.warning(f"[MOSS-BypassFix] Skipped runtime bypass LoRA QKV patch: {exc}")
        return False

    if getattr(comfy.sd.load_bypass_lora_for_models, "_moss_bypass_qkv_patched", False):
        return True

    def _resolve_submodule(model, module_key):
        module = model
        try:
            for part in module_key.split("."):
                if part.isdigit():
                    module = module[int(part)]
                else:
                    module = getattr(module, part)
            return module
        except (AttributeError, IndexError, KeyError):
            return None

    def _normalize_moss_lora_prefixes(lora):
        out = {}
        for k, v in lora.items():
            target_k = k
            if _MOSS_LAYER_PROJ_RE.search(k):
                for prefix in _MOSS_EXTRA_PREFIXES:
                    if k.startswith(prefix):
                        cand = "diffusion_model." + k[len(prefix):]
                        if cand not in lora and cand not in out:
                            target_k = cand
                        break
            out[target_k] = v
        return out

    class _MossMultiBypassHook:
        def __init__(self, module, entries):
            self.module = module
            self.entries = list(entries)
            self.original_forward = None
            self.original_force_forward = False
            self.module_info = get_module_type_info(module)

        def _configure_adapter(self, adapter, strength):
            info = self.module_info
            adapter.multiplier = strength
            adapter.is_conv = info["is_conv"]
            adapter.conv_dim = info["conv_dim"]
            adapter.kernel_size = info["kernel_size"]
            adapter.in_channels = info["in_channels"]
            adapter.out_channels = info["out_channels"]
            if info["is_conv"]:
                adapter.kw_dict = {
                    "stride": info["stride"],
                    "padding": info["padding"],
                    "dilation": info["dilation"],
                    "groups": info["groups"],
                }
            else:
                adapter.kw_dict = {}

        def _move_adapter_weights(self, adapter, device, dtype):
            if isinstance(adapter, torch.nn.Module):
                adapter.to(device=device)
                return
            weights = getattr(adapter, "weights", None)
            if weights is None:
                return
            if isinstance(weights, (list, tuple)):
                new_weights = []
                for w in weights:
                    if isinstance(w, torch.Tensor):
                        if dtype is not None and w.is_floating_point():
                            new_weights.append(w.to(device=device, dtype=dtype))
                        else:
                            new_weights.append(w.to(device=device))
                    else:
                        new_weights.append(w)
                adapter.weights = tuple(new_weights) if isinstance(weights, tuple) else new_weights
            elif isinstance(weights, torch.Tensor):
                if dtype is not None and weights.is_floating_point():
                    adapter.weights = weights.to(device=device, dtype=dtype)
                else:
                    adapter.weights = weights.to(device=device)

        def _ensure_adapter_device(self, adapter, device):
            if isinstance(adapter, torch.nn.Module):
                p = next(adapter.parameters(), None)
                if p is not None and p.device != device:
                    self._move_adapter_weights(adapter, device, None)
                return
            weights = getattr(adapter, "weights", None)
            first_tensor = None
            if isinstance(weights, (list, tuple)):
                for w in weights:
                    if isinstance(w, torch.Tensor):
                        first_tensor = w
                        break
            elif isinstance(weights, torch.Tensor):
                first_tensor = weights
            if first_tensor is not None and first_tensor.device != device:
                self._move_adapter_weights(adapter, device, None)

        def _bypass_forward(self, x, *args, **kwargs):
            if len(self.entries) == 1 and self.entries[0][2] is None:
                adapter, strength, _ = self.entries[0]
                self._configure_adapter(adapter, strength)
                self._ensure_adapter_device(adapter, x.device)
                adapter_bypass = getattr(adapter, "bypass_forward", None)
                if adapter_bypass is not None:
                    adapter_type = type(adapter)
                    is_default = (
                        adapter_type.bypass_forward is WeightAdapterBase.bypass_forward
                        or adapter_type.bypass_forward is WeightAdapterTrainBase.bypass_forward
                    )
                    if not is_default:
                        return adapter_bypass(self.original_forward, x, *args, **kwargs)

            base_out = self.original_forward(x, *args, **kwargs)
            out = None
            for adapter, strength, offset in self.entries:
                self._configure_adapter(adapter, strength)
                self._ensure_adapter_device(adapter, x.device)
                h_out = adapter.h(x, base_out)
                if offset is not None:
                    dim, start, length = offset
                    if out is None:
                        out = base_out.clone()
                    out_dim = (1 if getattr(adapter, "is_conv", False) else -1) if dim == 0 else dim
                    idx = [slice(None)] * out.ndim
                    idx[out_dim] = slice(start, start + length)
                    idx_t = tuple(idx)
                    out[idx_t] = out[idx_t] + h_out.to(device=out.device, dtype=out.dtype)
                else:
                    delta = h_out.to(device=base_out.device, dtype=base_out.dtype)
                    out = (base_out + delta) if out is None else (out + delta)
                    out = adapter.g(out)
            return base_out if out is None else out

        def inject(self):
            if self.original_forward is not None:
                return
            device = comfy.model_management.get_torch_device()
            dtype = None
            if hasattr(self.module, "weight") and self.module.weight is not None:
                cand_dtype = self.module.weight.dtype
                if cand_dtype in (torch.float32, torch.float16, torch.bfloat16):
                    dtype = cand_dtype
            for adapter, strength, _ in self.entries:
                self._configure_adapter(adapter, strength)
                self._move_adapter_weights(adapter, device, dtype)
            self.original_forward = self.module.forward
            self.original_force_forward = getattr(self.module, "comfy_force_forward", False)
            self.module.comfy_force_forward = True
            self.module.forward = self._bypass_forward

        def eject(self):
            if self.original_forward is None:
                return
            self.module.forward = self.original_forward
            self.module.comfy_force_forward = self.original_force_forward
            self.original_forward = None

    def _apply_bypass_to_patcher(patcher, target_model, bypass_patches, strength):
        model_sd_keys = set(target_model.state_dict().keys())
        matched = set()
        new_entries = []
        sliced_count = 0

        for raw_key, adapter in bypass_patches.items():
            if isinstance(raw_key, tuple):
                weight_key = raw_key[0]
                offset = raw_key[1] if len(raw_key) > 1 else None
                patch_fn = raw_key[2] if len(raw_key) > 2 else None
                if patch_fn is not None:
                    logging.warning(f"[MOSS-BypassFix] Unsupported custom patch function on key {weight_key}, skipping")
                    continue
            else:
                weight_key = raw_key
                offset = None

            if weight_key not in model_sd_keys:
                logging.warning(f"[BypassLoRA] Adapter key not in model state_dict: {raw_key}")
                continue

            module_key = weight_key[:-7] if weight_key.endswith(".weight") else weight_key
            new_entries.append((module_key, adapter, strength, offset))
            matched.add(raw_key)
            if offset is not None:
                sliced_count += 1

        prev_entries = []
        to_remove = []
        for inj_key, inj_list in list(patcher.injections.items()):
            if inj_key.startswith("bypass_lora"):
                has_moss = False
                for inj in inj_list:
                    stored = getattr(inj, "_moss_bypass_entries", None)
                    if stored is not None:
                        prev_entries.extend(stored)
                        has_moss = True
                if has_moss:
                    to_remove.append(inj_key)
        for inj_key in to_remove:
            patcher.remove_injections(inj_key)

        combined_entries = prev_entries + new_entries
        grouped = {}
        for mod_key, adapter, st, off in combined_entries:
            grouped.setdefault(mod_key, []).append((adapter, st, off))

        hooks = []
        for mod_key, mod_entries in grouped.items():
            module = _resolve_submodule(target_model, mod_key)
            if module is None or not hasattr(module, "weight"):
                logging.warning(f"[MOSS-BypassFix] Target module not found or missing weight: {mod_key}")
                continue
            hooks.append(_MossMultiBypassHook(module, mod_entries))

        if hooks:
            def inject_all(mp):
                for h in hooks:
                    h.inject()

            def eject_all(mp):
                for h in hooks:
                    h.eject()

            inj = PatcherInjection(inject=inject_all, eject=eject_all)
            inj._moss_bypass_entries = combined_entries
            patcher.set_injections(f"bypass_lora_{uuid.uuid4().hex[:8]}", [inj])

        return matched, len(new_entries), sliced_count, len(combined_entries)

    def _patched_load_bypass_lora_for_models(model, clip, lora, strength_model, strength_clip):
        key_map = {}
        if model is not None:
            key_map = comfy.lora.model_lora_keys_unet(model.model, key_map)
        if clip is not None:
            key_map = comfy.lora.model_lora_keys_clip(clip.cond_stage_model, key_map)

        lora = comfy.lora_convert.convert_lora(lora)
        lora = _normalize_moss_lora_prefixes(lora)
        loaded = comfy.lora.load_lora(lora, key_map)

        bypass_patches = {}
        regular_patches = {}
        for key, patch_data in loaded.items():
            if isinstance(patch_data, WeightAdapterBase):
                bypass_patches[key] = patch_data
            else:
                regular_patches[key] = patch_data

        k = set()
        k1 = set()

        if model is not None:
            new_modelpatcher = model.clone()
            if regular_patches:
                k.update(new_modelpatcher.add_patches(regular_patches, strength_model))
            matched_u, loaded_cnt, sliced_cnt, stacked_cnt = _apply_bypass_to_patcher(
                new_modelpatcher, new_modelpatcher.model, bypass_patches, strength_model
            )
            k.update(matched_u)
            if loaded_cnt > 0:
                logging.info(
                    f"[MOSS-BypassFix] Loaded {loaded_cnt} bypass adapters "
                    f"({sliced_cnt} sliced QKV), total stacked={stacked_cnt}"
                )
        else:
            new_modelpatcher = None

        if clip is not None:
            new_clip = clip.clone()
            if regular_patches:
                k1.update(new_clip.add_patches(regular_patches, strength_clip))
            matched_c, _, _, _ = _apply_bypass_to_patcher(
                new_clip.patcher, new_clip.cond_stage_model, bypass_patches, strength_clip
            )
            k1.update(matched_c)
        else:
            new_clip = None

        for x in loaded:
            if (x not in k) and (x not in k1):
                patch_data = loaded[x]
                patch_type = type(patch_data).__name__
                if isinstance(patch_data, tuple):
                    patch_type = f"tuple({patch_data[0]})"
                logging.warning(f"NOT LOADED: {x} (type={patch_type})")

        return (new_modelpatcher, new_clip)

    _patched_load_bypass_lora_for_models._moss_bypass_qkv_patched = True
    comfy.sd.load_bypass_lora_for_models = _patched_load_bypass_lora_for_models
    logging.info("[MOSS-BypassFix] Patched comfy.sd.load_bypass_lora_for_models for sliced QKV & multi-LoRA stacking")
    return True


_BYPASS_QKV_FIX_ACTIVE = _install_bypass_lora_qkv_fix()


def _get_drive_dir():
    return os.environ.get("MOSS_DRIVE_DATA_DIR", "/content/drive/MyDrive/ComfyUI_Data")


def _is_authorized(request):
    expected = os.environ.get("MOSS_BRIDGE_TOKEN", "").strip()
    provided = (request.headers.get("X-Bridge-Token") or "").strip()
    if not expected:
        return False
    return hmac.compare_digest(provided, expected)


def _is_allowed_host(host):
    h = (host or "").lower()
    return (
        h == "civitai.com"
        or h.endswith(".civitai.com")
        or h == "huggingface.co"
        or h.endswith(".huggingface.co")
        or h == "hf.co"
        or h.endswith(".hf.co")
    )


def _clear_comfy_model_cache():
    try:
        import folder_paths
        cache = getattr(folder_paths, "filename_list_cache", None)
        if isinstance(cache, dict):
            cache.clear()
    except Exception:
        pass


def _download_via_urllib(url, dest_path, headers=None):
    part_path = dest_path + ".part"
    req_headers = {"User-Agent": "MOSS-Image-Studio/1.0"}
    if headers:
        req_headers.update(headers)
    req = urllib.request.Request(url, headers=req_headers)
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            with open(part_path, "wb") as out:
                while True:
                    chunk = resp.read(1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
        if not os.path.isfile(part_path) or os.path.getsize(part_path) == 0:
            raise RuntimeError("Downloaded file is empty")
        os.replace(part_path, dest_path)
    except Exception:
        if os.path.exists(part_path):
            try:
                os.remove(part_path)
            except OSError:
                pass
        raise


def _sync_drive_subfolder_to_ssd(subfolder, filenames=None):
    drive_models = os.path.join(_get_drive_dir(), "models", subfolder)
    os.makedirs(drive_models, exist_ok=True)
    local_models_root = "/content/ComfyUI/models"
    ssd_dir = os.path.join(local_models_root, subfolder)
    has_ssd = os.path.isdir(local_models_root)
    if has_ssd:
        os.makedirs(ssd_dir, exist_ok=True)

    wanted_rel = set()
    wanted_base = set()
    if filenames:
        for item in filenames:
            norm = str(item).strip().replace("\\", "/").lstrip("/")
            if norm:
                wanted_rel.add(norm)
                wanted_base.add(os.path.basename(norm))

    copied = []
    skipped = []
    drive_files = []
    safety_margin = 5 * (1024 ** 3)

    for root, _, files in os.walk(drive_models):
        for fname in sorted(files):
            if fname.endswith(".aria2") or fname.endswith(".part"):
                continue
            if not fname.lower().endswith(ALLOWED_EXTS):
                continue
            src_file = os.path.join(root, fname)
            rel_path = os.path.relpath(src_file, drive_models).replace("\\", "/")
            drive_files.append(rel_path)
            if wanted_rel and rel_path not in wanted_rel and fname not in wanted_base:
                continue
            if not has_ssd:
                skipped.append(rel_path)
                continue
            dst_file = os.path.join(ssd_dir, rel_path)
            src_size = os.path.getsize(src_file)
            if os.path.isfile(dst_file) and os.path.getsize(dst_file) == src_size:
                skipped.append(rel_path)
                continue
            free_ssd = shutil.disk_usage("/content").free
            if free_ssd - src_size < safety_margin:
                skipped.append(rel_path)
                continue
            os.makedirs(os.path.dirname(dst_file), exist_ok=True)
            shutil.copy2(src_file, dst_file)
            copied.append(rel_path)

    _clear_comfy_model_cache()
    comfy_files = []
    try:
        import folder_paths
        comfy_files = list(folder_paths.get_filename_list(subfolder))
    except Exception:
        comfy_files = []

    return {
        "copied": copied,
        "skipped": skipped,
        "drive_files": sorted(drive_files),
        "comfy_files": comfy_files,
    }


async def _run_download(task_id, clean_url, subfolder, safe_filename, civitai_token, hf_token):
    task = DOWNLOAD_TASKS[task_id]
    drive_models = os.path.join(_get_drive_dir(), "models", subfolder)
    os.makedirs(drive_models, exist_ok=True)
    dest_drive = os.path.join(drive_models, safe_filename)
    aria2_ctrl = dest_drive + ".aria2"

    try:
        if os.path.isfile(dest_drive) and os.path.getsize(dest_drive) > 0 and not os.path.exists(aria2_ctrl):
            task["size_mb"] = round(os.path.getsize(dest_drive) / (1024 * 1024), 1)
            local_models_root = "/content/ComfyUI/models"
            if os.path.isdir(local_models_root):
                task["status"] = "syncing_ssd"
                await asyncio.to_thread(_sync_drive_subfolder_to_ssd, subfolder, [safe_filename])
            else:
                _clear_comfy_model_cache()
            task["status"] = "completed"
            return

        parsed = urlparse(clean_url)
        host = (parsed.hostname or "").lower()
        cmd = [
            "aria2c",
            "--console-log-level=warn",
            "--summary-interval=0",
            "--max-tries=3",
            "--retry-wait=2",
            "--user-agent=MOSS-Image-Studio/1.0",
            "-x", "8",
            "-s", "8",
            "-k", "1M",
            "-d", drive_models,
            "-o", safe_filename,
        ]
        if os.path.exists(aria2_ctrl):
            cmd.append("-c")
        else:
            cmd.extend(["--allow-overwrite=true", "--auto-file-renaming=false"])

        final_url = clean_url
        if (host == "civitai.com" or host.endswith(".civitai.com")) and civitai_token:
            qp = dict(parse_qsl(parsed.query))
            if "token" not in qp:
                qp["token"] = civitai_token
                final_url = urlunparse(parsed._replace(query=urlencode(qp)))
        req_headers = {"User-Agent": "MOSS-Image-Studio/1.0"}
        if (host == "huggingface.co" or host.endswith(".huggingface.co") or host == "hf.co") and hf_token:
            cmd.extend(["--header", f"Authorization: Bearer {hf_token}"])
            req_headers["Authorization"] = f"Bearer {hf_token}"
        cmd.append(final_url)

        proc = await asyncio.create_subprocess_exec(
            *cmd,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
        )
        stdout_b, stderr_b = await proc.communicate()
        if proc.returncode != 0 or not os.path.isfile(dest_drive) or os.path.getsize(dest_drive) == 0:
            combined = "\n".join(
                part.decode("utf-8", errors="ignore").strip()
                for part in (stderr_b, stdout_b)
                if part and part.decode("utf-8", errors="ignore").strip()
            ).strip()
            aria_err = combined or f"aria2c exited with {proc.returncode}"
            try:
                await asyncio.to_thread(_download_via_urllib, final_url, dest_drive, req_headers)
            except Exception as fb_exc:
                raise RuntimeError(f"{aria_err} | fallback failed: {fb_exc}")

        task["size_mb"] = round(os.path.getsize(dest_drive) / (1024 * 1024), 1)
        local_models_root = "/content/ComfyUI/models"
        if os.path.isdir(local_models_root):
            task["status"] = "syncing_ssd"
            await asyncio.to_thread(_sync_drive_subfolder_to_ssd, subfolder, [safe_filename])
        else:
            _clear_comfy_model_cache()

        task["status"] = "completed"
    except Exception as exc:
        task["status"] = "failed"
        task["error"] = str(exc)[:300]


routes = PromptServer.instance.routes


@routes.get("/moss_studio/ping")
async def moss_bridge_ping(request):
    return web.json_response({
        "ok": True,
        "drive_data_dir": _get_drive_dir(),
        "auth_enabled": bool(os.environ.get("MOSS_BRIDGE_TOKEN", "").strip()),
        "bridge_version": 3,
        "supports_sync_drive": True,
        "bypass_qkv_fix": bool(_BYPASS_QKV_FIX_ACTIVE),
    })


@routes.post("/moss_studio/sync_drive")
async def moss_bridge_sync_drive(request):
    if not _is_authorized(request):
        return web.json_response({"error": "Unauthorized bridge token"}, status=401)
    try:
        data = await request.json()
    except Exception:
        data = {}
    if not isinstance(data, dict):
        data = {}
    subfolder = str(data.get("subfolder") or "loras").strip()
    if subfolder not in ALLOWED_SUBFOLDERS:
        return web.json_response({"error": "Invalid subfolder"}, status=400)
    raw_names = data.get("filenames")
    filenames = None
    if isinstance(raw_names, list):
        filenames = [str(x).strip() for x in raw_names if str(x).strip()]
    result = await asyncio.to_thread(_sync_drive_subfolder_to_ssd, subfolder, filenames)
    return web.json_response({"ok": True, "subfolder": subfolder, **result})


@routes.post("/moss_studio/download")
async def moss_bridge_download(request):
    if not _is_authorized(request):
        return web.json_response({"error": "Unauthorized bridge token"}, status=401)
    try:
        data = await request.json()
    except Exception:
        return web.json_response({"error": "Invalid JSON"}, status=400)

    raw_url = str(data.get("url") or "").strip()
    parsed = urlparse(raw_url)
    if parsed.scheme != "https" or not _is_allowed_host(parsed.hostname):
        return web.json_response({"error": "Only https:// civitai.com or huggingface.co URLs are allowed"}, status=400)

    subfolder = str(data.get("subfolder") or "loras").strip()
    if subfolder not in ALLOWED_SUBFOLDERS:
        return web.json_response({"error": "Invalid subfolder"}, status=400)

    raw_name = str(data.get("filename") or "").strip()
    safe_name = Path(raw_name).name
    if not safe_name or safe_name != raw_name or ".." in safe_name or "/" in safe_name or "\\" in safe_name:
        return web.json_response({"error": "Invalid filename"}, status=400)
    if not safe_name.lower().endswith(ALLOWED_EXTS):
        return web.json_response({"error": "Unsupported model extension"}, status=400)

    civitai_token = str(data.get("civitai_token") or os.environ.get("MOSS_BRIDGE_TOKEN", "")).strip()
    hf_token = str(data.get("hf_token") or "").strip()
    task_id = uuid.uuid4().hex[:12]
    DOWNLOAD_TASKS[task_id] = {
        "task_id": task_id,
        "status": "downloading",
        "filename": safe_name,
        "subfolder": subfolder,
        "size_mb": 0.0,
        "error": None,
    }
    asyncio.create_task(_run_download(task_id, raw_url, subfolder, safe_name, civitai_token, hf_token))
    return web.json_response({"task_id": task_id, "filename": safe_name, "subfolder": subfolder})


@routes.get("/moss_studio/download/{task_id}")
async def moss_bridge_status(request):
    if not _is_authorized(request):
        return web.json_response({"error": "Unauthorized bridge token"}, status=401)
    tid = request.match_info["task_id"]
    task = DOWNLOAD_TASKS.get(tid)
    if not task:
        return web.json_response({"error": "Task not found"}, status=404)
    return web.json_response(task)
"""
with open(os.path.join(bridge_dir, "__init__.py"), "w", encoding="utf-8") as f:
    f.write(bridge_code)
print(f"✅ 已安装云端直传桥接插件: {bridge_dir}")

if COPY_MODELS_TO_LOCAL_SSD:
    import time
    drive_models_dir = os.path.join(DRIVE_DATA_DIR, "models")
    local_models_dir = os.path.join(COMFY_DIR, "models")
    if os.path.exists(drive_models_dir):
        print("\n[可选加速] 正在将云盘模型同步到 Colab 本地高速 SSD (/content/ComfyUI/models)...")
        safety_margin = 5 * (1024 ** 3)
        for root, _, files in os.walk(drive_models_dir):
            for fname in sorted(files):
                if fname.endswith(".aria2"):
                    continue
                src_file = os.path.join(root, fname)
                rel_path = os.path.relpath(src_file, drive_models_dir)
                dst_file = os.path.join(local_models_dir, rel_path)
                src_size = os.path.getsize(src_file)
                if os.path.exists(dst_file) and os.path.getsize(dst_file) == src_size:
                    print(f"  ⏭️ SSD 已存在，跳过: {rel_path}")
                    continue
                free_ssd = shutil.disk_usage("/content").free
                if free_ssd - src_size < safety_margin:
                    print(f"  ⚠️ 本地 SSD 剩余空间不足（留 5GB 缓冲），跳过本地拷贝，将直接从云盘读取: {rel_path}")
                    continue
                os.makedirs(os.path.dirname(dst_file), exist_ok=True)
                t0 = time.time()
                print(f"  🚀 正在拷贝到 SSD ({src_size / (1024 ** 3):.2f} GB): {rel_path} ...")
                shutil.copy2(src_file, dst_file)
                print(f"  ✅ 拷贝完成 ({time.time() - t0:.1f}s)")

print(f"\n✅ ComfyUI 安装并绑定完成！配置文件已写入: {yaml_path}")

In [ ]:
#@title 3. 一键下载 MOSS-Image-SFT-INT8 全套模型与官方工作流（支持自定义扩展下载）
import os
import shutil
import subprocess
import urllib.request
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

DRIVE_DATA_DIR = "/content/drive/MyDrive/ComfyUI_Data" #@param {type:"string"}
DOWNLOAD_MOSS_SFT_INT8_PACK = True #@param {type:"boolean"}
DOWNLOAD_SHINOBISAT_PACK = True #@param {type:"boolean"}
DOWNLOAD_MOSS_4STEP_TURBO_LORA = False #@param {type:"boolean"}
DOWNLOAD_SEEDVR2_MODELS = False #@param {type:"boolean"}
MOSS_TEXT_ENCODER = "qwen3.5_9b_int8_convrot.safetensors" #@param ["qwen3.5_9b_int8_convrot.safetensors", "qwen3.5_9b_w4a8_gptq.safetensors", "qwen3.5_9b_bf16.safetensors"]
MOSS_VAE = "flux2-vae.safetensors" #@param ["flux2-vae.safetensors", "flux2-vae_small.safetensors"]

# --- 可选：额外自定义模型下载 ---
MODEL_URL = "" #@param {type:"string"}
TARGET_SUBFOLDER = "diffusion_models" #@param ["checkpoints", "diffusion_models", "unet", "text_encoders", "clip", "clip_vision", "vae", "loras", "controlnet", "upscale_models", "embeddings"]
CUSTOM_FILENAME = "" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}
CIVITAI_TOKEN = "" #@param {type:"string"}

if not os.path.exists(DRIVE_DATA_DIR):
    raise RuntimeError(f"未检测到云盘目录 {DRIVE_DATA_DIR}，请先运行 Cell 1 挂载 Google Drive！")

if shutil.which("aria2c") is None:
    print("检测到尚未安装 aria2，正在自动安装（支持纯 CPU 下载环境）...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], check=True)


def get_secret_or_param(param_value, secret_name):
    if param_value and param_value.strip():
        return param_value.strip()
    try:
        from google.colab import userdata
        val = userdata.get(secret_name)
        return val.strip() if val else ""
    except Exception:
        return ""


RESOLVED_HF_TOKEN = get_secret_or_param(HF_TOKEN, "HF_TOKEN")
RESOLVED_CIVITAI_TOKEN = get_secret_or_param(CIVITAI_TOKEN, "CIVITAI_TOKEN")


def download_model(url, subfolder, filename=""):
    clean_url = url.strip()
    if not clean_url:
        return

    target_dir = os.path.join(DRIVE_DATA_DIR, "models", subfolder)
    os.makedirs(target_dir, exist_ok=True)

    clean_filename = filename.strip() if filename else ""
    if clean_filename:
        dest_file = os.path.join(target_dir, clean_filename)
        if os.path.exists(dest_file) and os.path.getsize(dest_file) > 0 and not os.path.exists(dest_file + ".aria2"):
            print(f"⏭️ 已存在于云盘，跳过重复下载: {clean_filename}")
            return

    parsed = urlparse(clean_url)
    host = parsed.netloc.lower()
    cmd = [
        "aria2c",
        "--console-log-level=error",
        "--summary-interval=10",
        "-c",
        "-x", "16",
        "-s", "16",
        "-k", "1M",
        "-d", target_dir,
    ]

    if ("huggingface.co" in host or host == "hf.co") and RESOLVED_HF_TOKEN:
        cmd.extend(["--header", f"Authorization: Bearer {RESOLVED_HF_TOKEN}"])

    if "civitai.com" in host and RESOLVED_CIVITAI_TOKEN:
        query_params = dict(parse_qsl(parsed.query))
        if "token" not in query_params:
            query_params["token"] = RESOLVED_CIVITAI_TOKEN
            clean_url = urlunparse(parsed._replace(query=urlencode(query_params)))

    if clean_filename:
        cmd.extend(["-o", clean_filename])

    cmd.append(clean_url)
    print(f"\n⬇️ 开始下载到: {target_dir}")
    subprocess.run(cmd, check=True)
    print(f"✅ 下载完成 ({subfolder})")


if DOWNLOAD_MOSS_SFT_INT8_PACK:
    print("=== 开始准备 MOSS-Image-SFT-INT8 全套运行文件 ===")
    download_model(
        "https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/diffusion_models/moss_image_sft_int8_convrot.safetensors",
        "diffusion_models",
        "moss_image_sft_int8_convrot.safetensors",
    )
    download_model(
        f"https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/text_encoders/{MOSS_TEXT_ENCODER}",
        "text_encoders",
        MOSS_TEXT_ENCODER,
    )
    download_model(
        f"https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/vae/{MOSS_VAE}",
        "vae",
        MOSS_VAE,
    )

    wf_dir = os.path.join(DRIVE_DATA_DIR, "user", "default", "workflows")
    os.makedirs(wf_dir, exist_ok=True)
    wf_path = os.path.join(wf_dir, "image_moss_image_t2i.json")
    if os.path.exists(wf_path) and os.path.getsize(wf_path) > 0:
        print(f"⏭️ 官方工作流已存在，跳过下载: {wf_path}")
    else:
        try:
            wf_url = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/image_moss_image_t2i.json"
            urllib.request.urlretrieve(wf_url, wf_path)
            print(f"✅ 官方工作流已保存到云盘: {wf_path}")
        except Exception as exc:
            print(f"⚠️ 工作流模板下载失败（不影响模型，进 ComfyUI 菜单 Templates 也能直接搜 MOSS-Image 打开）: {exc}")

if MODEL_URL.strip():
    download_model(MODEL_URL, TARGET_SUBFOLDER, CUSTOM_FILENAME)
elif not any([DOWNLOAD_MOSS_SFT_INT8_PACK, DOWNLOAD_SHINOBISAT_PACK, DOWNLOAD_MOSS_4STEP_TURBO_LORA, DOWNLOAD_SEEDVR2_MODELS]):
    print("ℹ️ 未勾选下载套件且 MODEL_URL 为空。请勾选上方下载项或填入下载直链。")

if DOWNLOAD_SHINOBISAT_PACK:
    print("\n=== 下载 ShinobiSat v2 玩家扩展包（8步加速 LoRA + Remacri 放大模型） ===")
    download_model(
        "https://huggingface.co/lodestones/Kromoss/resolve/main/8-steps-dmd2-lora.safetensors",
        "loras",
        "lodestones-8-steps-dmd2-lora.safetensors",
    )
    download_model(
        "https://huggingface.co/fofr/comfyui/resolve/main/upscale_models/4x_foolhardy_Remacri.pth",
        "upscale_models",
        "4x_foolhardy_Remacri.pth",
    )

if DOWNLOAD_MOSS_4STEP_TURBO_LORA:
    print("\n=== 下载 MOSS-Image 4-Step Turbo 加速 LoRA ===")
    download_model(
        "https://huggingface.co/akhaliq/MOSS-Image-SFT-4-steps/resolve/main/moss_image_sft_4step_v2.0.safetensors",
        "loras",
        "moss_image_sft_4step_v2.0.safetensors",
    )

if DOWNLOAD_SEEDVR2_MODELS:
    print("\n=== 下载 SeedVR2 2K/3K 超分模型 ===")
    download_model(
        "https://huggingface.co/numz/SeedVR2_comfyUI/resolve/main/seedvr2_ema_7b_sharp_fp16.safetensors",
        "diffusion_models",
        "seedvr2_ema_7b_sharp_fp16.safetensors",
    )
    download_model(
        "https://huggingface.co/Comfy-Org/SeedVR2/resolve/main/vae/ema_vae_fp16.safetensors",
        "vae",
        "ema_vae_fp16.safetensors",
    )

# ================= 额外批量下载示例（如 LoRA 等，按需取消注释） =================
# download_model("https://huggingface.co/Comfy-Org/MOSS-Image/resolve/main/loras/moss_image_color_depth_control.safetensors", "loras", "moss_image_color_depth_control.safetensors")

In [ ]:
#@title 4. 启动 ComfyUI 并开启公网访问隧道（Cloudflare / Localtunnel）
import json
import os
import re
import shlex
import shutil
import socket
import subprocess
import sys
import threading
import time
import urllib.request
from datetime import datetime, timezone

DRIVE_DATA_DIR = "/content/drive/MyDrive/ComfyUI_Data" #@param {type:"string"}
TUNNEL_TYPE = "cloudflared" #@param ["cloudflared", "localtunnel"]
BRIDGE_TOKEN = "" #@param {type:"string"}
EXTRA_ARGS = "" #@param {type:"string"}

COMFY_DIR = "/content/ComfyUI"
PORT = 8188
CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"
ENDPOINT_FILE = os.path.join(DRIVE_DATA_DIR, ".moss_studio_endpoint.json")

if not os.path.exists(os.path.join(COMFY_DIR, "main.py")):
    raise RuntimeError("未检测到 /content/ComfyUI/main.py，请先运行 Cell 2 安装 ComfyUI！")

for folder in ["output", "input", "user"]:
    os.makedirs(os.path.join(DRIVE_DATA_DIR, folder), exist_ok=True)


def _detect_gpu_name():
    if not shutil.which("nvidia-smi"):
        return ""
    try:
        res = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            timeout=5,
        )
        if res.returncode == 0:
            return res.stdout.strip().splitlines()[0].strip()
    except Exception:
        pass
    return ""


def _publish_endpoint(comfy_url, status="running", tunnel_type="cloudflared"):
    try:
        payload = {
            "comfy_url": comfy_url,
            "status": status,
            "tunnel_type": tunnel_type,
            "gpu_name": _detect_gpu_name(),
            "started_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "updated_at": int(time.time()),
        }
        tmp_path = ENDPOINT_FILE + ".tmp"
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        os.replace(tmp_path, ENDPOINT_FILE)
    except Exception as exc:
        print(f"⚠️ 同步公网地址到云盘失败（不影响手动复制使用）: {exc}")


published_url_holder = {"url": ""}

if TUNNEL_TYPE == "cloudflared":
    if not os.path.exists(CLOUDFLARED_BIN):
        print("正在下载 cloudflared...")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            CLOUDFLARED_BIN,
        )
        os.chmod(CLOUDFLARED_BIN, 0o755)
else:
    if shutil.which("lt") is None:
        print("正在安装 localtunnel...")
        subprocess.run(["npm", "install", "-g", "localtunnel"], check=True)

tunnel_proc = None


def tunnel_worker(port, mode):
    global tunnel_proc
    while True:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                break
        except OSError:
            time.sleep(1)

    print("\n🚀 ComfyUI 已就绪，正在建立公网访问隧道...\n")
    if mode == "cloudflared":
        tunnel_proc = subprocess.Popen(
            [CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        url_printed = False
        for line in tunnel_proc.stdout:
            if not url_printed:
                match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
                if match:
                    url = match.group(0)
                    published_url_holder["url"] = url
                    _publish_endpoint(url, "running", "cloudflared")
                    print("=" * 60)
                    print(f"👉 点击打开 ComfyUI: {url}")
                    print("📡 已将公网地址自动同步至 Google Drive，本地 MOSS-Image 工作台可一键自动连接！")
                    print("=" * 60)
                    url_printed = True
    else:
        try:
            ip = urllib.request.urlopen("https://ipv4.icanhazip.com", timeout=5).read().decode().strip()
            print(f"🔑 Localtunnel 访问密码 (Endpoint IP): {ip}")
        except Exception as exc:
            print(f"⚠️ 获取公网 IP 失败: {exc}")
        tunnel_proc = subprocess.Popen(
            ["lt", "--port", str(port)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        url_printed = False
        for line in tunnel_proc.stdout:
            if not url_printed and "http" in line:
                m_lt = re.search(r"https?://\S+", line.strip())
                lt_url = m_lt.group(0) if m_lt else line.strip()
                published_url_holder["url"] = lt_url
                _publish_endpoint(lt_url, "running", "localtunnel")
                print("=" * 60)
                print(f"👉 点击打开 ComfyUI: {lt_url}")
                print("📡 已将公网地址自动同步至 Google Drive，本地 MOSS-Image 工作台可一键自动连接！")
                print("=" * 60)
                url_printed = True


threading.Thread(target=tunnel_worker, args=(PORT, TUNNEL_TYPE), daemon=True).start()

launch_cmd = [
    sys.executable,
    "main.py",
    "--dont-print-server",
    "--output-directory", os.path.join(DRIVE_DATA_DIR, "output"),
    "--input-directory", os.path.join(DRIVE_DATA_DIR, "input"),
    "--user-directory", os.path.join(DRIVE_DATA_DIR, "user"),
]
if EXTRA_ARGS.strip():
    launch_cmd.extend(shlex.split(EXTRA_ARGS))

launch_env = os.environ.copy()
launch_env["MOSS_DRIVE_DATA_DIR"] = DRIVE_DATA_DIR
resolved_bridge_token = BRIDGE_TOKEN.strip()
if not resolved_bridge_token:
    try:
        from google.colab import userdata
        resolved_bridge_token = (userdata.get("CIVITAI_TOKEN") or "").strip()
    except Exception:
        resolved_bridge_token = ""
if not resolved_bridge_token:
    token_file = os.path.join(DRIVE_DATA_DIR, ".moss_bridge_token")
    if os.path.isfile(token_file):
        try:
            with open(token_file, "r", encoding="utf-8") as tf:
                resolved_bridge_token = tf.read().strip()
        except Exception:
            resolved_bridge_token = ""
if resolved_bridge_token:
    launch_env["MOSS_BRIDGE_TOKEN"] = resolved_bridge_token

drive_bridge_py = os.path.join(DRIVE_DATA_DIR, "moss_studio_bridge.py")
if os.path.isfile(drive_bridge_py):
    try:
        bridge_dst_dir = os.path.join(COMFY_DIR, "custom_nodes", "moss_studio_bridge")
        os.makedirs(bridge_dst_dir, exist_ok=True)
        shutil.copy2(drive_bridge_py, os.path.join(bridge_dst_dir, "__init__.py"))
        print("🔄 已从云盘 ComfyUI_Data/moss_studio_bridge.py 自动更新桥接与 Bypass LoRA 修复插件")
    except Exception as exc:
        print(f"⚠️ 自动更新 moss_studio_bridge 失败: {exc}")

comfy_proc = subprocess.Popen(
    launch_cmd,
    cwd=COMFY_DIR,
    env=launch_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    for log_line in comfy_proc.stdout:
        print(log_line, end="", flush=True)
    comfy_proc.wait()
finally:
    if published_url_holder["url"]:
        try:
            _publish_endpoint(published_url_holder["url"], "stopped", TUNNEL_TYPE)
        except Exception:
            pass
    if comfy_proc.poll() is None:
        comfy_proc.terminate()
    if tunnel_proc is not None and tunnel_proc.poll() is None:
        tunnel_proc.terminate()